# Writing a Composite

## What you'll learn

- How to define a `CompositeDefinition` with inputs, outputs, and parameters
- How `compose()` wires internal operations using `CompositeContext`
- How to run your composite with `pipeline.run_composite()`
- How to verify its outputs

**Prerequisites:** [Composites](../02-pipeline-design/07-composites.ipynb)
**Estimated time:** 20 minutes
**GPU required:** No.

---

Write a `CompositeDefinition` that generates, scores, and filters datasets.
Declare its public output and wire the internal operations in `compose()`.

In [ ]:
from __future__ import annotations

from enum import StrEnum
from typing import ClassVar

import polars as pl
from pydantic import BaseModel, Field

from artisan.composites import (
    CompositeContext,
    CompositeDefinition,
)
from artisan.operations.curator import Filter
from artisan.operations.examples import (
    DataGenerator,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager
from artisan.schemas import OutputSpec
from artisan.utils import tutorial_setup
from artisan.visualization import (
    build_macro_graph,
    build_micro_graph,
    inspect_pipeline,
)

## Choose the composite’s contract

This composite generates its own inputs, so callers only need parameters. It
will expose a `datasets` output containing the candidates that pass a score
threshold. Keep the internal metrics private: callers need the selected data,
while provenance still records how it was scored.

## Building `GenerateAndFilter`

Connect generation, scoring, and filtering:

```
DataGenerator ──→ MetricCalculator ──→ Filter
  (N datasets)      (N metrics)       (top datasets)
```

The complete class below declares that output and wires generation, scoring,
and filtering in `compose()`:

In [ ]:
class GenerateAndFilter(CompositeDefinition):
    """Generate datasets, score them, and filter by quality."""

    name = "generate_and_filter"
    description = "Generate, score, and quality-filter datasets."

    # External I/O roles
    class OutputRole(StrEnum):
        DATASETS = "datasets"

    # No InputRole — this is a generative composite (no external inputs)

    outputs: ClassVar[dict[str, OutputSpec]] = {
        OutputRole.DATASETS: OutputSpec(
            artifact_type="csv",
            description="Quality-filtered datasets",
        ),
    }

    # Composite-level parameters
    class Params(BaseModel):
        count: int = Field(default=6, description="Datasets to generate")
        threshold: float = Field(
            default=0.5, description="Minimum median score to keep"
        )
        seed: int = Field(default=42, description="Random seed")

    params: Params = Params()

    def compose(self, ctx: CompositeContext) -> None:
        # Step 1: Generate candidate datasets
        generated = ctx.run(
            DataGenerator,
            params={"count": self.params.count, "seed": self.params.seed},
        )

        # Step 2: Score each dataset (creates provenance edges for Filter)
        ctx.run(
            MetricCalculator,
            inputs={"dataset": generated.output("datasets")},
        )

        # Step 3: Filter — keep datasets above threshold
        filtered = ctx.run(
            Filter,
            inputs={"passthrough": generated.output("datasets")},
            params={
                "criteria": [
                    {
                        "metric": "distribution.median",
                        "operator": "ge",
                        "value": self.params.threshold,
                    }
                ],
            },
        )

        # Map internal result to declared output
        ctx.output("datasets", filtered.output("passthrough"))

`compose()` wires the operations with three methods:

- `ctx.run()` runs an operation and returns a `CompositeStepHandle`.
- `handle.output(role)` returns a `CompositeRef` for wiring inputs.
- `ctx.output(role, ref)` exposes an internal result as a composite output.

The `MetricCalculator` handle is unused because Filter discovers metrics through
provenance from its passthrough datasets. The criterion `distribution.median`
selects a nested metric key.

## Run the composite

`pipeline.run_composite()` expands the composite into real pipeline
steps — each internal `ctx.run()` becomes its own step with independent
caching and worker dispatch. Step names are prefixed with the composite
name.

In [ ]:
env = tutorial_setup("composite_run")

In [ ]:
pipeline = PipelineManager.create(
    name="writing_composite",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)

# Composites run via run_composite / submit_composite. Passing a
# CompositeDefinition to pipeline.run() raises TypeError.
selected = pipeline.run_composite(
    GenerateAndFilter,
    params={"count": 6, "threshold": 0.5, "seed": 42},
)

result = pipeline.finalize()

In [ ]:
from artisan.visualization import inspect_csv, inspect_step

assert result["overall_success"]
selected_output = selected.output("datasets")
assert selected_output.source_step == 2
accepted = inspect_step(
    env.delta_root,
    selected_output.source_step,
    pipeline_run_id=pipeline.config.pipeline_run_id,
)
assert 0 < accepted.height < 6
selected_data = inspect_csv(
    env.delta_root,
    step_number=selected_output.source_step,
    pipeline_run_id=pipeline.config.pipeline_run_id,
)
medians = selected_data.group_by("_source").agg(pl.col("score").median())
assert medians.height == accepted.height
assert (medians["score"] >= 0.5).all()
print(f"Verified {accepted.height} selected datasets meet the threshold")
inspect_pipeline(env.delta_root)

In [ ]:
build_macro_graph(env.delta_root)

The graph groups three steps under `generate_and_filter`: `data_generator`,
`metric_calculator`, and `filter`. The composite output contains the selected
datasets exposed by `ctx.output()`.

In [ ]:
build_micro_graph(env.delta_root)

## Next steps

- [Composites](../02-pipeline-design/07-composites.ipynb) — Reuse, override, and nest composites
- [Composites and Composition](../../concepts/composites-and-composition.md) — How composites expand
- [Writing Composite Operations](../../how-to-guides/writing-composite-operations.md) — Authoring recipes
- [Python API](../../reference/python-api.md) — Current composite method docstrings